> **Note:** This notebook requires a live OpenAI API key. Set `OPENAI_API_KEY` in your `.env` file before running. Smoke-run deferred — API key not available in CI.

# Multi-Step Agent with LangGraph `StateGraph`

`create_agent` is great for the standard tool-calling loop. Reach for **LangGraph `StateGraph`** when you need explicit control over a multi-step pipeline — branching, loops, custom routing.

Here we build a 3-node graph: **planner → tool → answer**.

**When to use which:** use `create_agent` for a single autonomous agent loop; use `StateGraph` when you want to hand-wire the steps and conditionally route between them.

In [ ]:
%pip install langgraph langchain-openai openai

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from openai import OpenAI

client = OpenAI()


class State(TypedDict):
    question: str
    plan: str
    tool_result: str
    answer: str

In [ ]:
def planner(state: State) -> dict:
    """Decide what lookup is needed to answer the question."""
    resp = client.responses.create(
        model="gpt-5.5",
        instructions="You are a planner. In one line, state what to look up to answer the question.",
        input=state["question"],
    )
    return {"plan": resp.output_text}

In [ ]:
def tool(state: State) -> dict:
    """Execute the lookup the plan asked for (stubbed search here)."""
    # In a real graph this would call a search API or database.
    resp = client.responses.create(
        model="gpt-5.5",
        tools=[{"type": "web_search"}],
        input=f"Find facts for this plan: {state['plan']}",
    )
    return {"tool_result": resp.output_text}

In [ ]:
def answer(state: State) -> dict:
    """Synthesize the final answer from the tool result."""
    resp = client.responses.create(
        model="gpt-5.5",
        instructions="Answer the user question using the provided research. Be concise.",
        input=f"Question: {state['question']}\nResearch: {state['tool_result']}",
    )
    return {"answer": resp.output_text}

In [ ]:
builder = StateGraph(State)
builder.add_node("planner", planner)
builder.add_node("tool", tool)
builder.add_node("answer", answer)

builder.add_edge(START, "planner")
builder.add_edge("planner", "tool")
builder.add_edge("tool", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

In [ ]:
result = graph.invoke({"question": "What is the population of Tokyo?"})
print("PLAN:  ", result["plan"])
print("ANSWER:", result["answer"])

In [ ]:
# Optional: conditional routing. Skip the tool when no lookup is needed.
def route_after_plan(state: State) -> str:
    if "no lookup" in state["plan"].lower():
        return "answer"
    return "tool"

cond_builder = StateGraph(State)
cond_builder.add_node("planner", planner)
cond_builder.add_node("tool", tool)
cond_builder.add_node("answer", answer)
cond_builder.add_edge(START, "planner")
cond_builder.add_conditional_edges("planner", route_after_plan)
cond_builder.add_edge("tool", "answer")
cond_builder.add_edge("answer", END)
cond_graph = cond_builder.compile()

In [ ]:
# Stream the graph to watch each node emit its update.
for chunk in graph.stream(
    {"question": "Who painted the Mona Lisa?"},
    stream_mode="updates",
):
    print(chunk)

## Recap

- `StateGraph(State)` + `add_node` / `add_edge` wires an explicit pipeline.
- `START` and `END` are the entry/exit sentinels; `compile()` validates the graph.
- `add_conditional_edges` enables branching — the main reason to choose LangGraph over `create_agent`.
- `stream(stream_mode="updates")` shows each node's delta; `"values"` shows full state.
- Each node returns a partial dict that is merged into state.